# CSV与JSON数据读写

学习目标：读写小型 CSV 与 JSON 文件，明确列类型、缺失值和日期的读取方式，并核对往返后的信息。

前置知识：Python 文件与路径、with 语句、列表与字典、DataFrame 和列选择。

运行环境：Python 3.12、pandas 3.0。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

示例使用自制小数据，后续单元沿用本章导入的名称。

本章使用默认 CSV 解析引擎和默认 JSON 解析引擎。pandas 3 的 str 列在本课程环境中由 PyArrow 存储；不设置 dtype_backend，也不把字符串存储后端与文件解析引擎混为一谈。

配套脚本：本章无外部脚本；[商品 CSV](date/02-products.csv) 是可直接查看的自制数据，数量单位为件，空字段为缺失，NA 是合法商品编号。

## 1 CSV 文件读写

CSV 用分隔符组织表格字段。read_csv() 将文件读成 DataFrame，to_csv() 将表格写回文件；下面先保存两条商品记录，再读取检查。

index=False 表示不额外导出行标签，encoding="utf-8" 指定文本编码。读写双方要约定相同编码，尤其是含中文的文件。

TemporaryDirectory 创建临时目录；Path 表示路径，斜杠连接完整文件名。本章在 with 块内完成文件读写，退出时自动删除目录及文件。传给 pandas 的是路径，本例读写结束后不保留打开的文件。

In [1]:
from io import StringIO
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd

products = pd.DataFrame({"name": ["铅笔", "本子"], "quantity": [3, 2]})

with TemporaryDirectory() as folder:
    csv_path = Path(folder) / "products.csv"
    products.to_csv(csv_path, index=False, encoding="utf-8")
    loaded = pd.read_csv(csv_path, encoding="utf-8")
    print(loaded)  # 两行两列，中文商品名完整；没有额外的索引数据列。
    print(loaded.shape)  # 预期：(2, 2)。

print(csv_path.exists())  # 预期：False，临时文件已经清理。
print(loaded["quantity"].tolist())  # 预期：[3, 2]，内存中的表格仍可使用。

  name  quantity
0   铅笔         3
1   本子         2
(2, 2)
False
[3, 2]


## 2 JSON 文件读写

同样的商品记录也可以用 JSON 交换。先用 orient="records" 把每一行写成“列名 → 值”的记录，再用相同的 orient 读回。

下面继续使用 products。force_ascii=False 让中文直接显示；records 保存记录数据，不保存原来的行索引。本表使用默认行号，读回后仍可按行号查看。

In [2]:
with TemporaryDirectory() as folder:
    json_path = Path(folder) / "products.json"
    products.to_json(json_path, orient="records", force_ascii=False)
    print(json_path.read_text(encoding="utf-8"))
    loaded = pd.read_json(
        json_path, orient="records", encoding="utf-8", convert_dates=False
    )
    print(loaded)
    print(loaded.columns.tolist(), loaded.shape)
    # 列名为 name、quantity；两行两列，铅笔数量为 3，本子为 2。

print(json_path.exists())  # False：内存中的 loaded 保留，临时文件已删除。

[{"name":"铅笔","quantity":3},{"name":"本子","quantity":2}]
  name  quantity
0   铅笔         3
1   本子         2
['name', 'quantity'] (2, 2)
False


## 3 分隔符与表头

sep 指定分隔符；默认按逗号分隔，并把首个非空行作为列名。输入没有表头时，用 header=None，再用 names 提供列名。

下面把两行分号分隔的文本交给 StringIO，作为内存中的文本流读取。with 结束时关闭文本流，不生成磁盘文件。

In [3]:
text = "铅笔;3\n本子;2\n"

with StringIO(text) as buffer:
    products = pd.read_csv(
        buffer, sep=";", header=None, names=["name", "quantity"]
    )

print(products)  # 首行仍是铅笔的记录，列名为 name、quantity。
print(products.shape)  # 预期：(2, 2)。

  name  quantity
0   铅笔         3
1   本子         2
(2, 2)


### 3.1 编码要与文件一致

文件中的字节要按写入时的编码读取。下面保存 UTF-8 中文，再故意按 ASCII 读取，捕获 UnicodeDecodeError；之后改用正确编码。不要用忽略解码错误的方法掩盖文本损坏。

In [4]:
products = pd.DataFrame({"name": ["铅笔"], "quantity": [3]})
with TemporaryDirectory() as folder:
    csv_path = Path(folder) / "products.csv"
    products.to_csv(csv_path, index=False, encoding="utf-8")
    try:
        pd.read_csv(csv_path, encoding="ascii", encoding_errors="strict")
    except UnicodeDecodeError as error:
        print(type(error).__name__)  # UnicodeDecodeError：ASCII 不能解码这里的中文。
    else:
        raise AssertionError("预期错误编码无法解码中文")
    loaded = pd.read_csv(csv_path, encoding="utf-8")

print(loaded["name"].tolist())  # ['铅笔']，正确编码恢复文本。

UnicodeDecodeError
['铅笔']


## 4 选择列与保留编号

只需要部分字段时，用 usecols 指定读取列；它不保证按参数列表排列列，需要顺序时再用列名列表选择。

编号 001 是文本标识。让读取器自动推断可能得到整数 1，丢掉前导零；dtype 可以逐列指定类型。本例用 str 保留编号，用 int64 保存整数数量。

dtype 决定列类型，不会关闭缺失值识别；编号可能与缺失标记同名时，还要约定缺失规则。

In [5]:
text = "code,name,quantity\n001,铅笔,3\n010,本子,2\n"

with StringIO(text) as buffer:
    inferred = pd.read_csv(buffer)
with StringIO(text) as buffer:
    selected = pd.read_csv(
        buffer, usecols=["quantity", "code"],
        dtype={"code": "str", "quantity": "int64"},
    )
selected = selected[["quantity", "code"]]

print(inferred["code"].tolist())  # 预期：[1, 10]，自动推断没有保留前导零。
print(selected)  # 列顺序为 quantity、code，编号仍为 001、010。
print(selected.dtypes)  # quantity 为 int64，code 为 str。

[1, 10]
   quantity code
0         3  001
1         2  010
quantity    int64
code          str
dtype: object


## 5 索引列

若记录编号本来就是行标签，可以在导出时保留索引，并用 index_label 为它命名。读回时，index_col 指定哪一列成为行标签。

CSV 中的字段不会自行说明“这是索引”。只保存数据列时用 index=False；保留业务行标签时，读写两端要同时约定。

In [6]:
stock = pd.DataFrame({"quantity": [3, 2]}, index=["R1", "R2"])

with TemporaryDirectory() as folder:
    csv_path = Path(folder) / "stock.csv"
    stock.to_csv(csv_path, index=True, index_label="record", encoding="utf-8")
    restored = pd.read_csv(csv_path, index_col="record", encoding="utf-8")

print(restored)  # R1、R2 是行标签；数据列仅有 quantity。
print(restored.index.name, restored.shape)  # 预期：record (2, 1)。

        quantity
record          
R1             3
R2             2
record (2, 1)


## 6 默认缺失标记

空字段、NA 等常见标记默认会被识别为缺失。isna 返回同形布尔结果，True 表示缺失；不要仅凭屏幕上显示的文字判断。

下面的 NA 是地区编号，而空字段才表示没有记录。先观察默认读取，再观察 keep_default_na=False 关闭默认清单后的结果。

In [7]:
text = "code,region\nA,NA\nB,\n"
with StringIO(text) as buffer:
    automatic = pd.read_csv(buffer)
with StringIO(text) as buffer:
    literal = pd.read_csv(buffer, keep_default_na=False)

print(automatic["region"].isna().tolist())  # [True, True]，两项都被识别为缺失。
print(literal["region"].tolist())  # ['NA', '']，两项都保留为文本。
print(literal["region"].isna().tolist())  # [False, False]，空字符串不是缺失值。

[True, True]
['NA', '']
[False, False]


## 7 指定缺失标记与日期

现在规定地区 NA 是普通文本，数量“缺测”才表示缺失。用 keep_default_na=False 关闭默认清单，再通过 na_values 按列指定缺失标记。

parse_dates 指定日期列，date_format 指定格式；这里 %Y、%m、%d 分别表示四位年份、月份、日期。日期解析后仍要检查 dtype，无法解析的列可能保留为文本。本例使用有效日期，数值缺失表示为 NaN。

In [8]:
text = "code,region,quantity,date\n001,NA,3,2026-09-01\n010,东区,缺测,2026-09-02\n"

with StringIO(text) as buffer:
    readings = pd.read_csv(
        buffer, dtype={"code": "str", "region": "str", "quantity": "float64"},
        keep_default_na=False, na_values={"quantity": ["缺测"]},
        parse_dates=["date"], date_format="%Y-%m-%d",
    )

print(readings)  # NA 保留为地区文本，第二条 quantity 为 NaN。
print(readings.dtypes)  # date 为日期时间类型，quantity 为 float64。
print(readings.isna())  # 只有第二行 quantity 为 True。

  code region  quantity       date
0  001     NA       3.0 2026-09-01
1  010     东区       NaN 2026-09-02
code                   str
region                 str
quantity           float64
date        datetime64[us]
dtype: object
    code  region  quantity   date
0  False   False     False  False
1  False   False      True  False


### 7.1 无法解析的日期列

parse_dates 是解析请求，不保证结果一定为日期时间类型。只要列中有无法按格式解析的值，read_csv 可能把整列保留为文本。因此导入后需要查看 dtype 和原始值，再决定如何处理错误记录。

In [9]:
text = "date,quantity\n2026-09-01,3\n未记录,2\n"
with StringIO(text) as buffer:
    unparsed = pd.read_csv(
        buffer, parse_dates=["date"], date_format="%Y-%m-%d"
    )

print(unparsed)
print(unparsed["date"].dtype)  # 本例为 str，而不是日期时间类型。
print(unparsed["date"].tolist())  # 原来的日期文本与“未记录”都保留。

         date  quantity
0  2026-09-01         3
1         未记录         2
str
['2026-09-01', '未记录']


## 8 错误行

字段过多的行默认触发解析错误。on_bad_lines="skip" 可以跳过这类行，但会减少记录，只有业务允许丢弃且已核对数量时才适合使用。

字段不足的行可能读成缺失值，不一定报错。解析成功也不能证明记录完整。下面分别观察字段过多与不足的输入。

In [10]:
too_many = "code,quantity\nA,3\nB,2,extra\nC,1\n"

try:
    with StringIO(too_many) as buffer:
        pd.read_csv(buffer, on_bad_lines="error")
except pd.errors.ParserError:
    print("捕获 ParserError：一行包含过多字段。")
else:
    raise AssertionError("预期字段过多触发 ParserError。")

with StringIO(too_many) as buffer:
    skipped = pd.read_csv(buffer, on_bad_lines="skip")
print(skipped)  # 仅剩 A、C，共两行；B 没有进入结果。

with StringIO("code,quantity\nA,3\nB\n") as buffer:
    short_row = pd.read_csv(buffer)
print(short_row)  # B 的 quantity 为 NaN，并未触发上面的错误。

捕获 ParserError：一行包含过多字段。
  code  quantity
0    A         3
1    C         1
  code  quantity
0    A       3.0
1    B       NaN


## 9 JSON 的表格结构

JSON 可以按记录或按标签组织同一张表。orient 指定组织方式，to_json() 与 read_json() 应使用相匹配的值。

| orient 值 | 中文名称／含义 |
| --- | --- |
| records | 记录列表，每条记录按列名保存值，不保存行索引 |
| split | 分别保存 index、columns、data |
| columns | 按列名组织，再按行标签组织值；DataFrame 的默认方式 |
| index | 按行标签组织，再按列名组织值 |
| values | 仅保存值数组，不保存行列标签 |
| table | 同时保存 schema 模式信息与记录数据 |

下面比较 records 与 split。未传输出路径时，to_json() 返回文本；force_ascii=False 让中文直接可读。用 StringIO 将 JSON 文本交给读取器，而不是把文本当作文件路径。

In [11]:
products = pd.DataFrame(
    {"code": ["001", "010"], "name": ["铅笔", "本子"]}, index=["A", "B"]
)
records_text = products.to_json(orient="records", force_ascii=False)
split_text = products.to_json(orient="split", force_ascii=False)

print(records_text)  # 记录列表含 code、name，不含 A、B 行标签。
print(split_text)  # 分别包含 index、columns、data。

with StringIO(records_text) as buffer:
    records_table = pd.read_json(
        buffer, orient="records", dtype={"code": "str"}, convert_dates=False
    )
with StringIO(split_text) as buffer:
    split_table = pd.read_json(
        buffer, orient="split", dtype={"code": "str"}, convert_dates=False
    )

print(records_table.index.tolist())  # 预期：[0, 1]。
print(split_table.index.tolist())  # 预期：['A', 'B']。
print(split_table.dtypes)  # 两列均为 str，编号仍保留前导零。

[{"code":"001","name":"铅笔"},{"code":"010","name":"本子"}]
{"columns":["code","name"],"index":["A","B"],"data":[["001","铅笔"],["010","本子"]]}
[0, 1]
['A', 'B']
code    str
name    str
dtype: object


### 9.1 其他 orient

用同一张两行一列的小表比较 columns、index、values 与 table。前两种把行列标签作为键；values 丢失行列标签；table 还包含描述字段的 schema。

下面读取非 table 结构时关闭日期和轴标签自动转换，便于专门观察结构。table 按自身 schema 读取；不能把它的模式信息理解为支持任意对象无损往返。

In [12]:
sample = pd.DataFrame({"quantity": [3, 2]}, index=["A", "B"])
for orient in ("columns", "index", "values", "table"):
    text = sample.to_json(orient=orient)
    with StringIO(text) as buffer:
        if orient == "table":
            restored = pd.read_json(buffer, orient=orient)
        else:
            restored = pd.read_json(
                buffer, orient=orient, convert_dates=False, convert_axes=False
            )
    print(orient, text)
    print(restored.index.tolist(), restored.columns.tolist(), restored.shape)
    print(restored)
    # 四种结果都是一列，数值按顺序为 3、2，形状都是 (2, 1)。
    # values 的标签变成 [0, 1] 与 [0]；其余三种保留 A、B 和 quantity。

columns {"quantity":{"A":3,"B":2}}
['A', 'B'] ['quantity'] (2, 1)
   quantity
A         3
B         2
index {"A":{"quantity":3},"B":{"quantity":2}}
['A', 'B'] ['quantity'] (2, 1)
   quantity
A         3
B         2
values [[3],[2]]
[0, 1] [0] (2, 1)
   0
0  3
1  2
table {"schema":{"fields":[{"name":"index","type":"string","extDtype":"str"},{"name":"quantity","type":"integer"}],"primaryKey":["index"],"pandas_version":"1.4.0"},"data":[{"index":"A","quantity":3},{"index":"B","quantity":2}]}
['A', 'B'] ['quantity'] (2, 1)
   quantity
A         3
B         2


### 9.2 标签唯一性

按标签作键的结构需要避免键冲突。DataFrame 使用 index 或 columns 结构时，行索引必须唯一；使用 index、columns 或 records 时，列名必须唯一。

下面只演示重复行标签。若需要保留这些重复标签，可选择把标签独立保存的 split；它会保留两条记录。

In [13]:
sample = pd.DataFrame({"quantity": [3, 2]}, index=["A", "A"])
try:
    sample.to_json(orient="index")
except ValueError as error:
    print(type(error).__name__)  # ValueError：不能把两个 A 同时作为唯一记录键。
else:
    raise AssertionError("预期 index 结构拒绝重复行标签")

text = sample.to_json(orient="split")
with StringIO(text) as buffer:
    restored = pd.read_json(buffer, orient="split", convert_dates=False)
print(restored.index.tolist(), restored.shape)  # ['A', 'A']，(2, 1)。

ValueError
['A', 'A'] (2, 1)


## 10 JSON Lines 与日期往返

JSON Lines 每行保存一条 JSON 记录，适合按记录交换数据。导出使用 orient="records" 与 lines=True，读取也设置 lines=True。

下面沿用“指定缺失标记与日期”中的 readings。to_json() 将缺失值写成 null；date_format="iso" 将日期写成 ISO 8601 文本。读回时用 convert_dates 指定日期列，keep_default_dates=False 避免根据其他列名猜测日期。

日期按 ISO 文本输出；date_unit 决定输出精度，本例显式使用毫秒 ms。只有日期的输入不受该精度选择影响，但更细的时间不能假定会完整保留。

In [14]:
with TemporaryDirectory() as folder:
    json_path = Path(folder) / "readings.jsonl"
    readings.to_json(
        json_path, orient="records", lines=True,
        force_ascii=False, date_format="iso", date_unit="ms",
    )
    print(json_path.read_text(encoding="utf-8"))
    # 两行 JSON；缺失数量写为 null，日期为带 T 的 ISO 文本。
    loaded = pd.read_json(
        json_path, orient="records", lines=True, encoding="utf-8",
        dtype={"code": "str", "quantity": "float64"},
        convert_dates=["date"], keep_default_dates=False,
    )

print(loaded)  # 两行四列；编号、中文地区、日期与缺失位置保留。
print(loaded.dtypes)  # 检查实际日期时间单位，不假定所有文件都恢复相同单位。
print(loaded["code"].tolist(), loaded["quantity"].isna().tolist())
# 编号为 001、010；数量缺失位置为 [False, True]。
print(loaded["date"].tolist() == readings["date"].tolist())  # True。
print(json_path.exists())  # False，文件已清理。

{"code":"001","region":"NA","quantity":3.0,"date":"2026-09-01T00:00:00.000"}
{"code":"010","region":"东区","quantity":null,"date":"2026-09-02T00:00:00.000"}



  code region  quantity       date
0  001     NA       3.0 2026-09-01
1  010     东区       NaN 2026-09-02
code                   str
region                 str
quantity           float64
date        datetime64[us]
dtype: object
['001', '010'] [False, True]
True
False


## 11 综合应用：约定 CSV 往返规则

继续使用 readings，将缺失数量写成“缺测”，日期写成年月日，编号保留为文本。先用默认参数读回，再按约定读取，对比类型与值。

CSV 没有保存 pandas 的完整类型信息；JSON 的 records 也没有保存行索引。选定格式后仍需检查标签、类型、日期精度和缺失位置，不能只看表格显示是否相似。

In [15]:
with TemporaryDirectory() as folder:
    csv_path = Path(folder) / "readings.csv"
    readings.to_csv(
        csv_path, index=False, encoding="utf-8",
        na_rep="缺测", date_format="%Y-%m-%d",
    )
    automatic = pd.read_csv(csv_path, encoding="utf-8")
    restored = pd.read_csv(
        csv_path, encoding="utf-8",
        dtype={"code": "str", "region": "str", "quantity": "float64"},
        keep_default_na=False, na_values={"quantity": ["缺测"]},
        parse_dates=["date"], date_format="%Y-%m-%d",
    )

print(automatic.dtypes)  # code 推断为整数；quantity、date 没有恢复原类型。
print(restored)  # 编号前导零、地区 NA、数量缺失和日期均按约定恢复。
print(restored.dtypes)  # 与原 readings 比较各列类型。
print(restored.columns.tolist(), restored.shape)
# 列顺序为 code、region、quantity、date，形状为 (2, 4)。
print(restored["code"].tolist(), restored["region"].tolist())
# 编号仍为 001、010，地区仍为 NA、东区。
print(restored["quantity"].isna().tolist())  # [False, True]。
print(restored["date"].tolist() == readings["date"].tolist())  # True。

code        int64
region        str
quantity      str
date          str
dtype: object


  code region  quantity       date
0  001     NA       3.0 2026-09-01
1  010     东区       NaN 2026-09-02
code                   str
region                 str
quantity           float64
date        datetime64[us]
dtype: object
['code', 'region', 'quantity', 'date'] (2, 4)
['001', '010'] ['NA', '东区']
[False, True]
True


### 11.1 小数精度也属于交换约定

JSON 保存数值时还受到输出精度限制。to_json 的 double_precision 默认是 10 位小数，最多可设为 15；这不是任意浮点数无损恢复的保证。

下面只观察一个短小数的序列化变化。需要何种精度由数据用途决定，不应把“读回成功”当作数值完全相同。

In [16]:
sample = pd.DataFrame({"value": [0.123456789012]})
default_text = sample.to_json(orient="records")
precise_text = sample.to_json(orient="records", double_precision=12)

print(default_text)  # 数值写成 0.123456789：末尾部分精度未保留。
print(precise_text)  # 本例写成 0.123456789012。
with StringIO(default_text) as buffer:
    restored = pd.read_json(buffer, orient="records", convert_dates=False)
print(restored["value"].tolist() == sample["value"].tolist())  # False。

[{"value":0.123456789}]
[{"value":0.123456789012}]
False


## 12 选学：展开嵌套记录

嵌套字典或记录列表不一定直接对应一张平面表。json_normalize() 接收已解码的 Python 对象，record_path 指出记录列表，meta 把外层字段带入每条记录，sep 指定嵌套字段名之间的分隔符。

下面将一个仓库的两条库存记录展开，保留仓库名称。

In [17]:
data = [{
    "warehouse": "北仓",
    "items": [
        {"code": "001", "stock": {"quantity": 3}},
        {"code": "010", "stock": {"quantity": 2}},
    ],
}]
flat = pd.json_normalize(data, record_path="items", meta=["warehouse"], sep=".")

print(flat)  # 两行三列；嵌套字段展开为 stock.quantity。
print(flat.shape)  # 预期：(2, 3)，两条记录都带有北仓。

  code  stock.quantity warehouse
0  001               3        北仓
1  010               2        北仓
(2, 3)


## 13 读取保留在课程目录中的 CSV

打开 date/02-products.csv 查看原始文本，再执行下面的读取。item_id 的前导零须保留；NA 在本文件中是合法编号，只有空字段表示缺失。数量使用可空整数 Int64，区分没有填写与真实的零。

这是独立的三行自制输入，不依赖前面临时文件的存在；读取不会删除或改写它。

In [18]:
file_products = pd.read_csv(
    "date/02-products.csv", encoding="utf-8",
    dtype={"item_id": "string", "name": "string", "quantity": "Int64"},
    keep_default_na=False, na_values=[""],
)
print(file_products)  # 编号为 001、010、NA；数量为 3、<NA>、0。
assert file_products["item_id"].tolist() == ["001", "010", "NA"]
assert file_products["quantity"].isna().tolist() == [False, True, False]
assert file_products.loc[2, "quantity"] == 0

  item_id name  quantity
0     001   铅笔         3
1     010   本子      <NA>
2      NA   橡皮         0

## 本章小结

（1）CSV 读写要约定分隔符、表头、索引、编码和类型；编号应在读取时保留为文本。

（2）缺失标记和日期解析需要明确参数，读完检查类型与缺失位置；字段不足也可能不报错。

（3）JSON 的 orient 决定标签与记录如何保存，JSON Lines 使用逐行记录。文本往返不保证自动恢复全部信息。

（4）文件示例在临时目录内完成并清理，内存中的 DataFrame 可以继续使用。应能解释一次往返中哪些信息由参数恢复。

## 练习

（1）读取下面没有表头的分号文本，列名为 code、quantity。编号必须保留前导零，并打印类型。

In [19]:
text = "007;4\n020;6\n"

# 在 with StringIO(text) as buffer 中读取，离开后打印结果。
# 检查：两行两列，code 为 str，编号为 007、020，数量为 4、6。
# 在此填写读取代码。

（2）先预测以下两次读取的 region 列，再运行核对。随后改为只把空字段当作缺失，保留 NA 文本，并在注释中解释参数选择理由。

In [20]:
text = "code,region\nA,NA\nB,\n"

with StringIO(text) as buffer:
    first = pd.read_csv(buffer)
with StringIO(text) as buffer:
    second = pd.read_csv(buffer, keep_default_na=False)

# 先记录预测，再比较两次读取的缺失位置。
print(first)
print(second)

# 在此完成“仅空字段为缺失”的读取，说明 na_values 与 keep_default_na 的组合。
# 检查：region 的第一行仍是 NA 文本，第二行是缺失值。

  code  region
0    A     NaN
1    B     NaN
  code region
0    A     NA
1    B       


（3）将下表分别转成 records 和 split JSON，再读回。任务要求保留行标签 A、B，选择合适的方式并说明理由。

In [21]:
sample = pd.DataFrame({"quantity": [4, 6]}, index=["A", "B"])

# 在此导出两种 JSON 文本，用 StringIO 读回并打印索引。
# 检查：两份结果的数据均为 4、6；判断哪种结构保存了原行标签。
# 在注释中解释你的选择。

（4）把下面的数据保存为 JSON Lines，再读回。所有文件操作放在 TemporaryDirectory 的 with 块内，完成后检查记录数与编号。

In [22]:
sample = pd.DataFrame({"code": ["007", "020"], "quantity": [4, 6]})

# 在此创建临时路径、导出并读回；编号列显式指定 str。
# 检查：文件有两条记录，读回 shape 为 (2, 2)，编号仍为 007、020。
# 退出 with 后检查临时路径不存在。

## 参考与引用来源

在线文档可能随发布更新；固定版本对照见下表 pandas v3.0.6 文档源码。API 参数与异常仍须结合所列页面的具体定位阅读。

| 网站 | 本章参考内容与定位 |
| --- | --- |
| pandas 官方在线文档（课程基线 3.0.6） | [read_csv](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) 的 sep、header、names、index_col、usecols、dtype、na_values、keep_default_na、parse_dates、date_format、encoding、on_bad_lines；[to_csv](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html) 的 index、index_label、na_rep、encoding、date_format；[read_json](https://pandas.pydata.org/docs/reference/api/pandas.read_json.html) 的 orient、dtype、convert_dates、keep_default_dates、convert_axes、lines 与标签唯一性条件；[to_json](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_json.html) 的 orient、force_ascii、date_format、date_unit、lines、double_precision 和缺失值说明；[String dtype migration](https://pandas.pydata.org/docs/user_guide/migration-3-strings.html) 的 Background、Brief introduction：默认 str 与 PyArrow 存储；[isna](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isna.html) 的缺失检查；[Series.to_list](https://pandas.pydata.org/docs/reference/api/pandas.Series.to_list.html) 的转列表观察；[json_normalize](https://pandas.pydata.org/docs/reference/api/pandas.json_normalize.html) 的 record_path、meta、sep 与 Examples。 |
| Python 官方文档（Python 3.12） | [TemporaryDirectory](https://docs.python.org/3.12/library/tempfile.html#tempfile.TemporaryDirectory) 的上下文退出与自动清理；[StringIO](https://docs.python.org/3.12/library/io.html#io.StringIO) 的内存文本流；[IOBase](https://docs.python.org/3.12/library/io.html#io.IOBase) 的上下文关闭；[pathlib](https://docs.python.org/3.12/library/pathlib.html) 的 Operators、Path.exists、Path.read_text；[日期格式代码](https://docs.python.org/3.12/library/datetime.html#strftime-and-strptime-format-codes) 的 %Y、%m、%d。 |
| GitHub 官方项目（版本化来源） | pandas v3.0.6 文档源码：[migration-3-strings](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/migration-3-strings.rst)；对应上列同名指南或发布说明的小节，作为固定版本对照。 |